1) Bank System (Magic Methods + Static + Passing Objects)
Build a BankAccount class.
Requirements
Constructor: owner, balance, currency
Static/class tracking:
total accounts created
a registry of all active accounts (store objects safely)
Magic methods:
__repr__ shows owner + masked account id + balance
__len__ returns number of transactions
__bool__ returns True only if account is active and balance > 0
__add__ allows acc1 + acc2 only if same currency → returns new account object combining balances (owner becomes "Joint")
__call__(amount, note="") acts like deposit/withdraw based on sign

Outside-class functions:
freeze_account(acc) receives an object, freezes it, returns the same object
transfer(sender, receiver, amount) returns a tuple of updated objects (sender, receiver) and raises custom exceptions on invalid cases

Hard constraints
No direct balance modification from outside; force method usage.
Transactions should be stored and affect __len__.

In [1]:
class InsufficientFundsError(Exception):
    pass

class AccountFrozenError(Exception):
    pass


class BankAccount:
    total_accounts = 0          # total created ever
    active_accounts = 0         # currently active (not frozen/closed)
    _next_id = 1000             # internal id generator

    def __init__(self, owner: str, balance: float, currency: str):
        if balance < 0:
            raise ValueError("Starting balance cannot be negative.")
        self.owner = owner
        self._balance = float(balance)      # “protected” (use property)
        self.currency = currency.upper()
        self.frozen = False
        self.closed = False
        self.account_id = BankAccount._next_id
        BankAccount._next_id += 1

        self._transactions = []  # for __len__

        BankAccount.total_accounts += 1
        BankAccount.active_accounts += 1

    # ---------- Properties ----------
    @property
    def balance(self) -> float:
        return self._balance

    # ---------- Magic methods ----------
    def __repr__(self) -> str:
        masked = f"****{str(self.account_id)[-2:]}"
        return f"BankAccount(owner={self.owner!r}, id={masked}, balance={self.balance:.2f} {self.currency})"

    def __len__(self) -> int:
        return len(self._transactions)

    def __bool__(self) -> bool:
        return (not self.frozen) and (not self.closed) and (self.balance > 0)

    def __add__(self, other):
        if not isinstance(other, BankAccount):
            return NotImplemented
        if self.currency != other.currency:
            raise ValueError("Cannot combine accounts with different currencies.")
        # returns a NEW account object (as required)
        joint = BankAccount("Joint", self.balance + other.balance, self.currency)
        joint._transactions.append(("CREATE_JOINT", self.balance, other.balance))
        return joint

    def __call__(self, amount: float, note: str = ""):
        # deposit/withdraw based on sign
        amount = float(amount)
        if amount == 0:
            return self
        if amount > 0:
            return self.deposit(amount, note=note)
        else:
            return self.withdraw(-amount, note=note)

    # ---------- Core behaviors ----------
    def _ensure_active(self):
        if self.closed:
            raise ValueError("Account is closed.")
        if self.frozen:
            raise AccountFrozenError("Account is frozen.")

    def deposit(self, amount: float, note: str = ""):
        self._ensure_active()
        if amount <= 0:
            raise ValueError("Deposit amount must be > 0.")
        self._balance += amount
        self._transactions.append(("DEPOSIT", amount, note))
        return self  # returning the object is useful for chaining

    def withdraw(self, amount: float, note: str = ""):
        self._ensure_active()
        if amount <= 0:
            raise ValueError("Withdraw amount must be > 0.")
        if amount > self._balance:
            raise InsufficientFundsError("Insufficient funds.")
        self._balance -= amount
        self._transactions.append(("WITHDRAW", amount, note))
        return self

    def close(self):
        if not self.closed:
            self.closed = True
            BankAccount.active_accounts -= 1
            self._transactions.append(("CLOSE", 0, ""))
        return self

    def freeze(self):
        self._ensure_active()
        self.frozen = True
        BankAccount.active_accounts -= 1
        self._transactions.append(("FREEZE", 0, ""))
        return self

    def unfreeze(self):
        if self.closed:
            raise ValueError("Closed accounts cannot be unfrozen.")
        if self.frozen:
            self.frozen = False
            BankAccount.active_accounts += 1
            self._transactions.append(("UNFREEZE", 0, ""))
        return self


# ---------- Outside-class functions ----------
def freeze_account(acc: BankAccount) -> BankAccount:
    return acc.freeze()

def transfer(sender: BankAccount, receiver: BankAccount, amount: float, note: str = ""):
    if not isinstance(sender, BankAccount) or not isinstance(receiver, BankAccount):
        raise TypeError("sender and receiver must be BankAccount objects.")
    if sender.currency != receiver.currency:
        raise ValueError("Currency mismatch for transfer.")
    if amount <= 0:
        raise ValueError("Transfer amount must be > 0.")

    sender.withdraw(amount, note=f"TRANSFER_OUT: {note}")
    receiver.deposit(amount, note=f"TRANSFER_IN: {note}")
    return sender, receiver


# ---------- Quick test (optional) ----------
if __name__ == "__main__":
    a = BankAccount("Ali", 500, "usd")
    b = BankAccount("Sara", 300, "usd")

    print(a)
    print(b)

    transfer(a, b, 100, note="Rent split")
    print(a, "transactions:", len(a))
    print(b, "transactions:", len(b))

    j = a + b
    print("Joint:", j)
    print("bool(a):", bool(a))

    a(-50, "snacks")   # withdraw using __call__
    a(200, "salary")   # deposit using __call__
    print(a, "transactions:", len(a))

BankAccount(owner='Ali', id=****00, balance=500.00 USD)
BankAccount(owner='Sara', id=****01, balance=300.00 USD)
BankAccount(owner='Ali', id=****00, balance=400.00 USD) transactions: 1
BankAccount(owner='Sara', id=****01, balance=400.00 USD) transactions: 1
Joint: BankAccount(owner='Joint', id=****02, balance=800.00 USD)
bool(a): True
BankAccount(owner='Ali', id=****00, balance=550.00 USD) transactions: 3


2) Shapes Engine (Inheritance + Polymorphism + Magic Comparisons)
Create abstract-ish base class Shape.
Requirements
Shape has constructor with name
Every child must implement:
area() and perimeter()
Implement at least: Circle, Rectangle, Triangle
Polymorphism task:
Write a function largest_shape(shapes) that accepts a list of Shape objects and returns the object with the largest area.
Magic methods:

__lt__, __eq__ compare shapes by area, but:
If areas equal, compare by perimeter
If still equal, compare by name
__hash__ must work so shapes can be put in a set (be careful with mutability)
Outside-class function:

scale(shape, factor) returns a new scaled object of the same type without modifying the original

Edge cases

Triangle validation in constructor (raise error if invalid sides)

Floating point comparisons: don’t use direct equality; apply tolerance.


In [1]:
from abc import ABC, abstractmethod
import math

class Shape(ABC):
    @abstractmethod
    def Area(self):
        pass

    @abstractmethod
    def Perimeter(self):
        pass

    def __lt__(self, other):
        return self.Area() < other.Area()

    def __repr__(self):
        return f"{self.__class__.__name__} (Area={self.Area():.2f})"


class Rectangle(Shape):
    def __init__(self, length, width):
        self.l = length
        self.w = width

    def Area(self):
        return self.l * self.w

    def Perimeter(self):
        return 2 * (self.l + self.w)


class Circle(Shape):
    def __init__(self, radius):
        self.r = radius

    def Area(self):
        return math.pi * self.r ** 2

    def Perimeter(self):
        return 2 * math.pi * self.r


class Triangle(Shape):
    def __init__(self, a, b, c):
        self.a = a
        self.b = b
        self.c = c

    def Perimeter(self):
        return self.a + self.b + self.c

    def Area(self):
        s = self.Perimeter() / 2
        return math.sqrt(s * (s - self.a) * (s - self.b) * (s - self.c))


def largest_shape(shapes):
    return max(shapes)


shapes = [
    Rectangle(4, 5),
    Circle(3),
    Triangle(3, 4, 5)
]

print(largest_shape(shapes))

Circle (Area=28.27)


In [2]:
# Problem-1: Class inheritence
# Create a Bus child class that inherits from the Vehicle class. The default fare charge of any vehicle is seating capacity * 100. If Vehicle is Bus instance, we need to add an extra 10% on full fare as a maintenance charge. So total fare for bus instance will become the final amount = total fare + 10% of the total fare.

# Note: The bus seating capacity is 50. so the final fare amount should be 5500. You need to override the fare() method of a Vehicle class in Bus class.
class Vehicle:
    def __init__(self, name, seating_capacity):
        self.name = name
        self.seating_capacity = seating_capacity

    def fare(self):
        return self.seating_capacity * 100


class Bus(Vehicle):
    def fare(self):
        base_fare = super().fare()
        maintenance = base_fare * 0.10
        return base_fare + maintenance

Vehicle.seating_capacity=50
bus = Bus("School Volvo", 50)
print("Final bus fare:", bus.fare())

Final bus fare: 5500.0


In [1]:
# Problem-2: Class Inheritence
# Create a Bus class that inherits from the Vehicle class. Give the capacity argument of Bus.seating_capacity() a default value of 50.

# Use the following code for your parent Vehicle class.
class Vehicle:
    def __init__(self, name):
        self.name = name

    def seating_capacity(self, capacity):
        return f"The seating capacity of {self.name} is {capacity} passengers"


class Bus(Vehicle):

    def seating_capacity(self, capacity=50):
        return super().seating_capacity(capacity)


bus = Bus("School Volvo")

print(bus.seating_capacity())

The seating capacity of School Volvo is 50 passengers


In [3]:
class Point:
    def __init__(self, x=0, y=0):
        self.x = x
        self.y = y


class Location:
    def __init__(self, location, destination):
        self.location = location
        self.destination = destination

    def reflect_destination_on_x_axis(self):
        reflected_x = self.destination.x
        reflected_y = -self.destination.y
        print(f"Reflection of Destination on x-axis: ({reflected_x}, {reflected_y})")


p1 = Point(2, 3)
p2 = Point(5, 7)

obj = Location(p1, p2)
obj.reflect_destination_on_x_axis()

Reflection of Destination on x-axis: (5, -7)


In [4]:
from abc import ABC, abstractmethod


class Polygon(ABC):
    @abstractmethod
    def get_data(self):
        pass

    @abstractmethod
    def area(self):
        pass


class Rectangle(Polygon):
    def get_data(self):
        self.length = float(input("Enter rectangle length: "))
        self.width = float(input("Enter rectangle width: "))

    def area(self):
        return self.length * self.width


class Triangle(Polygon):
    def get_data(self):
        self.base = float(input("Enter triangle base: "))
        self.height = float(input("Enter triangle height: "))

    def area(self):
        return 0.5 * self.base * self.height


r = Rectangle()
r.get_data()
print("Area of rectangle:", r.area())

t = Triangle()
t.get_data()
print("Area of triangle:", t.area())

Area of rectangle: 144.0
Area of triangle: 72.0


In [5]:
class Bill:
    def __init__(self, amount):
        self.amount = amount

    def pay(self):
        pass


class CashPayment(Bill):
    def pay(self):
        print(f"Bill of {self.amount} paid by cash.")


class ChequePayment(Bill):
    def pay(self):
        print(f"Bill of {self.amount} paid by cheque.")


cash = CashPayment(5000)
cash.pay()

cheque = ChequePayment(8000)
cheque.pay()

Bill of 5000 paid by cash.
Bill of 8000 paid by cheque.


In [6]:
class FlexibleDict(dict):
    def __getitem__(self, key):
        if key in self:
            return super().__getitem__(key)

        alt_key = str(key) if not isinstance(key, str) else self._convert_to_number(key)

        if alt_key in self:
            return super().__getitem__(alt_key)

        raise KeyError(key)

    def _convert_to_number(self, key):
        try:
            return int(key)
        except ValueError:
            return key


d = FlexibleDict()
d[1] = "One"
d["2"] = "Two"

print(d[1])     # One
print(d["1"])   # One
print(d[2])     # Two
print(d["2"])   # Two

One
One
Two
Two
